# Notebook 3 — CNN + TCN (Temporal Convolutional Network) for Parkinson's Detection

**Model:** CNN + TCN for EEG-based neurological disorder detection.

- **Disorder:** Parkinson's Disease (PD)
- **Dataset:** OpenNeuro ds004584 (Resting-state EEG)
- **Input:** Preprocessed EEG epochs shaped `(N, 60, 500)` — 60 channels, 2s @ 250Hz
- **Output:** Binary classification — PD (1) vs Control (0)
- **Data split:** Subject-wise Train/Val/Test (70/15/15)

> Uses preprocessed data from `data/processed/ds004584_cnn_tcn/`

In [ ]:
# ===== Imports =====
import os
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:

# ===== Config =====
@dataclass
class Config:
    # Data paths
    data_dir: str = "../data/processed/ds004584_cnn_tcn"

    # Model parameters (updated after loading data)
    n_channels: int = 60
    n_classes: int = 2
    sampling_rate: int = 250
    epoch_seconds: float = 2.0
    n_samples: int = 500

    # Training
    batch_size: int = 32
    lr: float = 5e-4
    weight_decay: float = 5e-4
    epochs: int = 3
    patience: int = 12
    grad_clip_norm: float = 1.0

    # Model
    # IMPROVED: dropout 0.50→0.35 and wider TCN channels 32→64
    dropout: float = 0.35
    tcn_channels: int = 64  # wider TCN (was 32)
    label_smoothing: float = 0.05

    # Data augmentation
    use_augmentation: bool = True
    noise_std: float = 0.10
    time_mask_ratio: float = 0.12
    channel_drop_prob: float = 0.08
    time_shift_max: int = 20

    # Mixup
    use_mixup: bool = True
    mixup_alpha: float = 0.2

    # Loss + class imbalance
    # IMPROVED: focal loss + explicit PD weight boost
    use_focal_loss: bool = True
    focal_gamma: float = 2.0
    pd_weight_boost: float = 2.5

    # Weighted sampler
    use_weighted_sampler: bool = True
    sampler_pd_boost: float = 2.0

    # Threshold tuning settings
    min_pd_recall: float = 0.75
    min_control_recall: float = 0.65


cfg = Config()
print(f"dropout={cfg.dropout}, tcn_channels={cfg.tcn_channels}")
print(f"use_focal_loss={cfg.use_focal_loss}, focal_gamma={cfg.focal_gamma}, pd_weight_boost={cfg.pd_weight_boost}")
print(f"use_weighted_sampler={cfg.use_weighted_sampler}, sampler_pd_boost={cfg.sampler_pd_boost}")
print(f"epochs={cfg.epochs}, patience={cfg.patience}, batch={cfg.batch_size}, lr={cfg.lr}")
print(f"Target: PD Recall >={cfg.min_pd_recall:.0%}, Control Recall >={cfg.min_control_recall:.0%}")


## Data Loading

Loading preprocessed subject-wise split data from `data/processed/ds004584_cnn_tcn/`:
- `train.npz`, `val.npz`, `test.npz` — Subject-wise split data
- Each file: `X` (N, 60, 500), `y` (N,), `subject_id` (N,)
- Already z-score normalized (stats from training set only)

In [ ]:
# ===== Load preprocessed data =====
def load_split(data_dir: str, split: str):
    """Load a data split (train/val/test) from .npz file."""
    path = Path(data_dir) / f"{split}.npz"
    if not path.exists():
        raise FileNotFoundError(
            f"Expected preprocessed data at {path}.\n"
            "Run preprocessing/preprocess_ds004584_cnn_tcn.py first."
        )
    data = np.load(path, allow_pickle=True)
    X = data["X"].astype(np.float32)   # (N, C, T)
    y = data["y"].astype(np.int64)     # (N,)
    subject_id = data["subject_id"] if "subject_id" in data.files else None
    return X, y, subject_id

# Load all splits
print("Loading preprocessed data...")
# Resolve data directory robustly (handles different notebook working directories)
data_path = Path(cfg.data_dir)
if not (data_path / "train.npz").exists():
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([
            base / "data" / "processed" / "ds004584_cnn_tcn",
            base / "code" / "data" / "processed" / "ds004584_cnn_tcn",
            base / "code" / "Parkinsons" / "data" / "processed" / "ds004584_cnn_tcn",
        ])
    found = next((p for p in candidates if (p / "train.npz").exists()), None)
    if found is None:
        raise FileNotFoundError(
            f"Could not find train.npz for cfg.data_dir='{cfg.data_dir}' from cwd='{Path.cwd()}'.\n"
            "Run preprocessing/preprocess_ds004584_cnn_tcn.py first."
        )
    cfg.data_dir = str(found)
    print(f"Resolved data_dir to: {cfg.data_dir}")

X_train, y_train, sid_train = load_split(cfg.data_dir, "train")
X_val, y_val, sid_val = load_split(cfg.data_dir, "val")
X_test, y_test, sid_test = load_split(cfg.data_dir, "test")

# Update config with actual dimensions
cfg.n_channels = X_train.shape[1]
cfg.n_samples = X_train.shape[2]

print(f"\n{'='*50}")
print("DATASET LOADED")
print(f"{'='*50}")
print(f"Train: {X_train.shape} | PD: {(y_train==1).sum()}, Control: {(y_train==0).sum()}")
print(f"Val:   {X_val.shape} | PD: {(y_val==1).sum()}, Control: {(y_val==0).sum()}")
print(f"Test:  {X_test.shape} | PD: {(y_test==1).sum()}, Control: {(y_test==0).sum()}")
print(f"\nChannels: {cfg.n_channels}, Samples: {cfg.n_samples}")

In [ ]:

# ===== EEG Data Augmentation =====
class EEGAugmentation:
    """EEG-specific data augmentation (training only)."""
    def __init__(self, noise_std=0.1, time_mask_ratio=0.1, channel_drop_prob=0.0, time_shift_max=0):
        self.noise_std = noise_std
        self.time_mask_ratio = time_mask_ratio
        self.channel_drop_prob = channel_drop_prob
        self.time_shift_max = time_shift_max

    def add_gaussian_noise(self, x):
        return x + torch.randn_like(x) * self.noise_std

    def time_masking(self, x):
        T = x.shape[-1]
        mask_len = int(T * self.time_mask_ratio)
        if mask_len > 0:
            start = torch.randint(0, T - mask_len, (1,)).item()
            x = x.clone()
            x[..., start:start + mask_len] = 0
        return x

    def channel_dropout(self, x):
        if self.channel_drop_prob <= 0:
            return x
        C = x.shape[-2]
        drop_mask = torch.rand(C, device=x.device) < self.channel_drop_prob
        if drop_mask.any():
            x = x.clone()
            x[drop_mask, :] = 0
        return x

    def time_shift(self, x):
        if self.time_shift_max <= 0:
            return x
        shift = int(torch.randint(-self.time_shift_max, self.time_shift_max + 1, (1,)).item())
        if shift == 0:
            return x
        return torch.roll(x, shifts=shift, dims=-1)

    def __call__(self, x):
        if torch.rand(1).item() < 0.5:
            x = self.add_gaussian_noise(x)
        if torch.rand(1).item() < 0.5:
            x = self.time_masking(x.clone())
        if torch.rand(1).item() < 0.5:
            x = self.channel_dropout(x)
        if torch.rand(1).item() < 0.5:
            x = self.time_shift(x)
        return x


# ===== Torch Dataset =====
class EEGDataset(Dataset):
    def __init__(self, X, y, augment=False, aug_config=None):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
        self.augment = augment
        self.augmenter = EEGAugmentation(
            noise_std=aug_config.noise_std if aug_config else 0.1,
            time_mask_ratio=aug_config.time_mask_ratio if aug_config else 0.1,
            channel_drop_prob=aug_config.channel_drop_prob if aug_config else 0.0,
            time_shift_max=aug_config.time_shift_max if aug_config else 0
        ) if augment else None

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment and self.augmenter:
            x = self.augmenter(x)
        return x, self.y[idx]


# Create datasets
train_dataset = EEGDataset(X_train, y_train, augment=cfg.use_augmentation, aug_config=cfg)
val_dataset = EEGDataset(X_val, y_val, augment=False)
test_dataset = EEGDataset(X_test, y_test, augment=False)

# IMPROVED: WeightedRandomSampler to oversample PD in training batches
from torch.utils.data import WeightedRandomSampler
if cfg.use_weighted_sampler:
    class_counts = np.bincount(y_train)
    weights_np = 1.0 / np.maximum(class_counts, 1).astype(float)
    weights_np[1] *= cfg.sampler_pd_boost
    sample_weights = weights_np[y_train]
    train_sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, sampler=train_sampler, drop_last=False)
    print(f"WeightedRandomSampler active (PD boost={cfg.sampler_pd_boost}x, counts={class_counts.tolist()})")
else:
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, drop_last=False)

val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, drop_last=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


## Model: CNN + TCN (Temporal Convolutional Network)

**Architecture:**
1. **CNN front-end** — Temporal conv (1×25) → Spatial conv (C×1) extracts spatiotemporal features
2. **TCN backbone** — 3 dilated causal TCN blocks (dilation 1, 2, 4) capture multi-scale temporal patterns
3. **Classifier** — Global average pooling → Linear layer for class logits + confidence head

**Advantages:** Lightweight, no recurrence, parallelizable, effective for EEG temporal dynamics

In [ ]:

# ===== CNN + TCN Model (improved: multi-scale front + wider/deeper TCN) =====
class TCNBlock(nn.Module):
    """Temporal Convolutional Network block with residual connection."""
    def __init__(self, in_ch, out_ch, k=3, dilation=1, dropout=0.2):
        super().__init__()
        pad = (k - 1) * dilation  # causal padding
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=k, dilation=dilation, padding=pad)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=k, dilation=dilation, padding=pad)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.down = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        y = self.drop(self.act(self.bn1(self.conv1(x))))
        y = self.drop(self.act(self.bn2(self.conv2(y))))
        if y.shape[-1] != x.shape[-1]:
            y = y[..., :x.shape[-1]]
        return self.act(y + self.down(x))


class CNN_TCN(nn.Module):
    """
    Improved CNN + TCN for EEG classification.

    Changes from baseline:
    1. Multi-scale CNN front-end: two parallel temporal branches
       - Broad (k=51): slow rhythms — theta/alpha (~4-13 Hz), PD slow-wave biomarkers
       - Narrow (k=13): fast rhythms  — beta (~13-30 Hz), PD beta hyper-synchrony
    2. Wider TCN channels: 32 → tcn_channels (default: 64)
    3. Deeper TCN: 3 blocks → 4 blocks (dilations 1, 2, 4, 8)
       Receptive field grows from ~28 to ~56 time steps (224ms @ 250Hz)
    4. feat_dim exposed for ECN compatibility
    """
    def __init__(self, n_channels, n_classes, T, dropout=0.35, tcn_channels=64):
        super().__init__()

        out_per_branch = 12  # broad + narrow, each 12 filters

        # Block 1a: broad temporal branch → slow EEG rhythms
        self.broad_conv = nn.Sequential(
            nn.Conv2d(1, out_per_branch, kernel_size=(1, 51), padding=(0, 25), bias=False),
            nn.BatchNorm2d(out_per_branch), nn.ELU()
        )
        # Block 1b: narrow temporal branch → fast EEG rhythms
        self.narrow_conv = nn.Sequential(
            nn.Conv2d(1, out_per_branch, kernel_size=(1, 13), padding=(0, 6), bias=False),
            nn.BatchNorm2d(out_per_branch), nn.ELU()
        )

        # Block 2: spatial convolution (projects 24 multi-scale channels → tcn_channels)
        ms_channels = out_per_branch * 2  # 24
        self.spatial = nn.Sequential(
            nn.Conv2d(ms_channels, tcn_channels, kernel_size=(n_channels, 1), bias=False),
            nn.BatchNorm2d(tcn_channels), nn.ELU()
        )

        # Block 3: pooling + dropout
        self.pool = nn.AvgPool2d(kernel_size=(1, 2))
        self.drop2d = nn.Dropout(dropout)

        # Block 4: TCN backbone — 4 dilated blocks, receptive field ~224ms
        tcn_drop = dropout / 2
        self.tcn = nn.Sequential(
            TCNBlock(tcn_channels, tcn_channels, k=3, dilation=1, dropout=tcn_drop),
            TCNBlock(tcn_channels, tcn_channels, k=3, dilation=2, dropout=tcn_drop),
            TCNBlock(tcn_channels, tcn_channels, k=3, dilation=4, dropout=tcn_drop),
            TCNBlock(tcn_channels, tcn_channels, k=3, dilation=8, dropout=tcn_drop),  # NEW
        )

        # Block 5: classifier head
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(tcn_channels, n_classes)
        self.conf_head = nn.Linear(tcn_channels, 1)
        self.feat_dim = tcn_channels  # exposed for ECN compatibility

    def forward(self, x, return_features=False):
        x = x.unsqueeze(1)                          # (B, C, T) → (B, 1, C, T)

        # Multi-scale temporal features
        broad = self.broad_conv(x)
        narrow = self.narrow_conv(x)
        t_min = min(broad.shape[-1], narrow.shape[-1])
        x = torch.cat([broad[..., :t_min], narrow[..., :t_min]], dim=1)  # (B, 24, C, T)

        x = self.spatial(x)                          # (B, tcn_channels, 1, T)
        x = self.pool(x)                             # (B, tcn_channels, 1, T//2)
        x = self.drop2d(x)
        x = x.squeeze(2)                             # (B, tcn_channels, T//2)
        x = self.tcn(x)                              # (B, tcn_channels, T//2)
        feats = self.gap(x).squeeze(-1)              # (B, tcn_channels)
        logits = self.classifier(feats)
        conf = torch.sigmoid(self.conf_head(feats))
        if return_features:
            return logits, conf, feats
        return logits


# Instantiate model
T = cfg.n_samples
model = CNN_TCN(cfg.n_channels, cfg.n_classes, T=T, dropout=cfg.dropout, tcn_channels=cfg.tcn_channels).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Feature dimension (feat_dim): {model.feat_dim}")


In [ ]:

# ===== Training & Evaluation Utilities =====
def mixup_data(x, y, alpha=0.0):
    """Apply Mixup to a batch."""
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def train_one_epoch(model, loader, optimizer, criterion, mixup_alpha=0.0, grad_clip=1.0):
    """Train for one epoch with optional mixup and gradient clipping."""
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()

        if mixup_alpha > 0:
            xb, y_a, y_b, lam = mixup_data(xb, yb, mixup_alpha)
            logits = model(xb)
            loss = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
        else:
            logits = model(xb)
            loss = criterion(logits, yb)

        loss.backward()
        if grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)


def eval_model(model, loader):
    """Evaluate model on a data loader."""
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            y_true.append(yb.numpy())
            y_pred.append(pred)
            y_probs.append(probs.cpu().numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_probs = np.concatenate(y_probs)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='binary')
    cm = confusion_matrix(y_true, y_pred)
    return acc, f1, cm, y_true, y_pred, y_probs


## Training Loop

- Uses subject-wise **Train/Val/Test** split (no cross-validation leakage)
- Early stopping based on validation F1 score
- Saves best model checkpoint
- Class-weighted loss to handle PD/Control imbalance

In [ ]:

# ===== Training: Focal Loss + WeightedRandomSampler + Gradient Clipping =====

# --- Focal Loss ---
class FocalLoss(nn.Module):
    """Focal Loss: down-weights easy examples, focuses on hard-to-classify ones."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        fl = (1 - pt) ** self.gamma * ce
        if self.alpha is not None:
            fl = self.alpha[targets] * fl
        return fl.mean()

# --- Class weights with PD boost ---
class_counts_arr = np.bincount(y_train)
cw_np = len(y_train) / (len(class_counts_arr) * class_counts_arr.astype(float))
cw_np[1] *= cfg.pd_weight_boost  # Boost PD class weight
class_weights = torch.FloatTensor(cw_np).to(DEVICE)
print(f"Class weights → Control={class_weights[0]:.3f}, PD={class_weights[1]:.3f}")

# --- Loss function ---
if cfg.use_focal_loss:
    criterion = FocalLoss(alpha=class_weights, gamma=cfg.focal_gamma)
    print(f"Using Focal Loss (gamma={cfg.focal_gamma})")
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=cfg.label_smoothing)
    print("Using CrossEntropyLoss")

# --- Fresh model ---
model = CNN_TCN(cfg.n_channels, cfg.n_classes, T=cfg.n_samples,
                dropout=cfg.dropout, tcn_channels=cfg.tcn_channels).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=7)

best_val_f1 = -1
best_state = None
patience_counter = 0

print(f"\nTraining CNN-TCN for {cfg.epochs} epochs (patience={cfg.patience})...")
print(f"{'='*70}")

for epoch in range(1, cfg.epochs + 1):
    mixup_alpha = cfg.mixup_alpha if cfg.use_mixup else 0.0
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion,
                                 mixup_alpha=mixup_alpha, grad_clip=cfg.grad_clip_norm)

    val_acc, val_f1, val_cm, _, _, _ = eval_model(model, val_loader)

    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_f1)
    new_lr = optimizer.param_groups[0]['lr']

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        marker = " *** best ***"
    else:
        patience_counter += 1
        marker = ""

    lr_info = f" (lr→{new_lr:.1e})" if new_lr < old_lr else ""
    if epoch % 2 == 0 or epoch == 1 or marker or lr_info:
        print(f"Epoch {epoch:03d} | loss={train_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}{marker}{lr_info}")

    if patience_counter >= cfg.patience:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {cfg.patience} epochs)")
        break

# Load best model
model.load_state_dict(best_state)
model = model.to(DEVICE)
print(f"\nBest validation F1: {best_val_f1:.4f}")

# Save best model
save_dir = "../training/saved_models"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "cnn_tcn_parkinsons.pth")
torch.save(best_state, save_path)
print(f"Model saved to: {save_path}")


## Test Set Evaluation

Final evaluation on the held-out test set (unseen subjects).
Reports accuracy, F1-score, confusion matrix, and per-class metrics.

In [ ]:

# ===== Test Set Evaluation with Threshold Tuning =====
import numpy as np

print("=" * 70)
print("TEST SET EVALUATION (CNN-TCN)")
print("=" * 70)

# --- Tune threshold on validation set ---
model.eval()
val_probs_all, val_labels_all = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb.to(DEVICE))
        val_probs_all.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        val_labels_all.extend(yb.numpy())
val_probs_all = np.array(val_probs_all)
val_labels_all = np.array(val_labels_all)

best_thr = 0.5
best_bal = -1
for thr in np.arange(0.20, 0.70, 0.01):
    preds_v = (val_probs_all >= thr).astype(int)
    pd_mask = val_labels_all == 1
    ctrl_mask = val_labels_all == 0
    pd_rec = np.mean(preds_v[pd_mask] == 1) if pd_mask.any() else 0.0
    ctrl_rec = np.mean(preds_v[ctrl_mask] == 0) if ctrl_mask.any() else 0.0
    bal = 0.5 * (pd_rec + ctrl_rec)
    if pd_rec >= 0.75 and ctrl_rec >= 0.65 and bal > best_bal:
        best_bal = bal
        best_thr = thr

print(f"Optimal threshold (val): {best_thr:.2f}")

# --- Collect test probabilities ---
_, _, _, test_y_true, test_y_pred, test_y_probs_arr = eval_model(model, test_loader)
test_probs_pd = test_y_probs_arr[:, 1]

# --- Report at both thresholds ---
for thr, label in [(0.5, "default (0.50)"), (best_thr, f"tuned  ({best_thr:.2f})")]:
    y_pred_t = (test_probs_pd >= thr).astype(int)
    t_acc = accuracy_score(test_y_true, y_pred_t)
    t_f1 = f1_score(test_y_true, y_pred_t)
    t_cm = confusion_matrix(test_y_true, y_pred_t)
    tn, fp, fn, tp = t_cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    print(f"\n[Threshold = {label}]")
    print(f"  Accuracy={t_acc:.4f}  F1={t_f1:.4f}")
    print(f"  Sensitivity (PD Recall)={sensitivity:.4f}   Specificity (Ctrl Recall)={specificity:.4f}")
    print(f"  Confusion Matrix:")
    print(f"                  Predicted")
    print(f"                  Control   PD")
    print(f"  Actual Control   {tn:5d}  {fp:5d}")
    print(f"  Actual PD        {fn:5d}  {tp:5d}")

# Best threshold classification report
optimal_preds = (test_probs_pd >= best_thr).astype(int)
print(f"\n{'='*70}")
print("CLASSIFICATION REPORT (tuned threshold)")
print(f"{'='*70}")
print(classification_report(test_y_true, optimal_preds, target_names=["Control (0)", "PD (1)"], digits=4)) 


## Hybrid Model: Base CNN-TCN + Error Correction Network (ECN)

Train an ECN on top of the best base model checkpoint.

- **Base model** predicts `PD vs Control` from EEG features
- **ECN** learns a residual correction over the base model's features, softmax probabilities, and confidence score
- **Final prediction** uses `base_logits + residual_logits`
- Base model weights are **frozen** during ECN training

In [ ]:

# ===== ECN: Definition + Utilities =====
class ErrorCorrectionNetwork(nn.Module):
    """Residual correction head over frozen base-model features and confidence.

    Input:  [base_features (feat_dim) | softmax_probs (n_classes) | confidence (1)]
    Output: residual logits added to base logits → hybrid logits
    """
    def __init__(self, feat_dim, n_classes=2, hidden_dim=256, dropout=0.20):
        super().__init__()
        input_dim = feat_dim + n_classes + 1
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_classes),
        )

    def forward(self, x):
        return self.net(x)


def _forward_hybrid(base_model, ecn_model, xb):
    base_logits, base_conf, base_feats = base_model(xb, return_features=True)
    base_probs = torch.softmax(base_logits, dim=1)
    ecn_input = torch.cat([base_feats, base_probs, base_conf], dim=1)
    residual_logits = ecn_model(ecn_input)
    hybrid_logits = base_logits + residual_logits
    return base_logits, hybrid_logits, residual_logits


def train_hybrid_ecn(base_model, ecn_model, train_loader, val_loader, cw_tensor,
                     epochs=15, lr=3e-4, residual_penalty=5e-4, patience=5):
    """Train ECN while keeping base_model frozen."""
    base_model.eval()
    for p in base_model.parameters():
        p.requires_grad = False

    optimizer = optim.AdamW(ecn_model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=cw_tensor.detach())

    best_state = None
    best_val_f1 = -1.0
    patience_ctr = 0
    history = {"train_loss": [], "val_loss": [], "val_f1": []}

    print("=" * 60)
    print("HYBRID TRAINING (ECN over frozen base)")
    print("=" * 60)

    for epoch in range(1, epochs + 1):
        ecn_model.train()
        train_loss = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()

            with torch.no_grad():
                base_logits, base_conf, base_feats = base_model(xb, return_features=True)
                base_probs = torch.softmax(base_logits, dim=1)
                ecn_input = torch.cat([base_feats, base_probs, base_conf], dim=1)

            residual_logits = ecn_model(ecn_input)
            hybrid_logits = base_logits + residual_logits

            cls_loss = criterion(hybrid_logits, yb)
            reg_loss = residual_penalty * residual_logits.pow(2).mean()
            loss = cls_loss + reg_loss
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= max(1, len(train_loader))

        ecn_model.eval()
        val_loss = 0.0
        val_preds_list, val_labels_list = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                _, hybrid_logits, residual_logits = _forward_hybrid(base_model, ecn_model, xb)
                cls_loss = criterion(hybrid_logits, yb)
                reg_loss = residual_penalty * residual_logits.pow(2).mean()
                val_loss += (cls_loss + reg_loss).item()
                val_preds_list.extend(torch.argmax(hybrid_logits, dim=1).cpu().numpy())
                val_labels_list.extend(yb.cpu().numpy())

        val_loss /= max(1, len(val_loader))
        val_f1 = f1_score(val_labels_list, val_preds_list)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in ecn_model.state_dict().items()}
            patience_ctr = 0
            marker = " *** BEST ***"
        else:
            patience_ctr += 1
            marker = ""

        print(f"ECN Epoch {epoch:03d} | Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f} | Val F1={val_f1:.4f}{marker}")

        if patience_ctr >= patience:
            print(f"Early stopping ECN at epoch {epoch}")
            break

    if best_state is not None:
        ecn_model.load_state_dict(best_state)
    print(f"Best ECN Val F1: {best_val_f1:.4f}")
    return ecn_model, history


def collect_pd_probs(base_model, loader, ecn_model=None):
    """Collect PD class probabilities from base or hybrid model."""
    base_model.eval()
    if ecn_model is not None:
        ecn_model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            if ecn_model is None:
                logits = base_model(xb)
            else:
                _, logits, _ = _forward_hybrid(base_model, ecn_model, xb)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(yb.numpy())
    return np.array(all_labels), np.array(all_probs)


def tune_threshold_from_probs(labels, probs, min_pd_recall=0.75, min_control_recall=0.65):
    """Find classification threshold maximising balanced accuracy subject to recall constraints."""
    labels = np.asarray(labels)
    probs = np.asarray(probs)
    results = []
    for thresh in np.arange(0.20, 0.70, 0.01):
        preds = (probs >= thresh).astype(int)
        pd_mask = labels == 1
        ctrl_mask = labels == 0
        pd_recall = np.mean(preds[pd_mask] == 1) if np.any(pd_mask) else 0.0
        ctrl_recall = np.mean(preds[ctrl_mask] == 0) if np.any(ctrl_mask) else 0.0
        f1 = f1_score(labels, preds)
        bal = 0.5 * (pd_recall + ctrl_recall)
        results.append((thresh, f1, pd_recall, ctrl_recall, bal))
    feasible = [r for r in results if r[2] >= min_pd_recall and r[3] >= min_control_recall]
    best = max(feasible if feasible else results, key=lambda r: r[4])
    return best[0], best[1], best[2], best[3]


print("ECN utilities defined: ErrorCorrectionNetwork, train_hybrid_ecn, collect_pd_probs, tune_threshold_from_probs")


In [ ]:

# ===== Train ECN on top of best base checkpoint =====
# Reload the best base model weights
model.load_state_dict(best_state)
model.eval()

ecn_model = ErrorCorrectionNetwork(
    feat_dim=model.feat_dim,
    n_classes=cfg.n_classes,
    hidden_dim=256,
    dropout=0.20,
).to(DEVICE)

ecn_model, ecn_history = train_hybrid_ecn(
    model,
    ecn_model,
    train_loader,
    val_loader,
    cw_tensor=class_weights,
    epochs=max(15, cfg.epochs // 2),
    lr=3e-4,
    residual_penalty=5e-4,
    patience=1, # 5 > 1
)
print("ECN training complete.")
print(f"ECN parameters: {sum(p.numel() for p in ecn_model.parameters()):,}")


In [ ]:

# ===== Base vs Hybrid Evaluation (ECN) =====
val_labels_base, val_probs_base = collect_pd_probs(model, val_loader, ecn_model=None)
val_labels_hyb, val_probs_hyb = collect_pd_probs(model, val_loader, ecn_model=ecn_model)

base_thr_ecn, base_val_f1_ecn, base_val_pd_rec, base_val_ctrl_rec = tune_threshold_from_probs(
    val_labels_base, val_probs_base,
    min_pd_recall=cfg.min_pd_recall, min_control_recall=cfg.min_control_recall,
)
hyb_thr_ecn, hyb_val_f1_ecn, hyb_val_pd_rec, hyb_val_ctrl_rec = tune_threshold_from_probs(
    val_labels_hyb, val_probs_hyb,
    min_pd_recall=cfg.min_pd_recall, min_control_recall=cfg.min_control_recall,
)

print("=" * 60)
print("VALIDATION THRESHOLD TUNING")
print("=" * 60)
print(f"Base   threshold={base_thr_ecn:.2f} | F1={base_val_f1_ecn:.4f} | PD recall={base_val_pd_rec:.4f} | Ctrl recall={base_val_ctrl_rec:.4f}")
print(f"Hybrid threshold={hyb_thr_ecn:.2f} | F1={hyb_val_f1_ecn:.4f} | PD recall={hyb_val_pd_rec:.4f} | Ctrl recall={hyb_val_ctrl_rec:.4f}")

# Test set evaluation
test_labels_base, test_probs_base_v = collect_pd_probs(model, test_loader, ecn_model=None)
test_labels_hyb, test_probs_hyb_v = collect_pd_probs(model, test_loader, ecn_model=ecn_model)

test_preds_base_v = (test_probs_base_v >= base_thr_ecn).astype(int)
test_preds_hyb_v = (test_probs_hyb_v >= hyb_thr_ecn).astype(int)

base_acc_ecn = accuracy_score(test_labels_base, test_preds_base_v)
base_f1_ecnr = f1_score(test_labels_base, test_preds_base_v)
hyb_acc_ecn = accuracy_score(test_labels_hyb, test_preds_hyb_v)
hyb_f1_ecnr = f1_score(test_labels_hyb, test_preds_hyb_v)

base_wrong = test_preds_base_v != test_labels_base
hyb_wrong = test_preds_hyb_v != test_labels_hyb
rescued = int(np.sum(base_wrong & (~hyb_wrong)))
hurt = int(np.sum((~base_wrong) & hyb_wrong))

test_cm_base_ecn = confusion_matrix(test_labels_base, test_preds_base_v)
test_cm_hyb_ecn = confusion_matrix(test_labels_hyb, test_preds_hyb_v)

print("\n" + "=" * 60)
print("TEST RESULTS: BASE vs HYBRID (ECN)")
print("=" * 60)
print(f"Base   | Acc={base_acc_ecn:.4f} | F1={base_f1_ecnr:.4f}")
print(f"Hybrid | Acc={hyb_acc_ecn:.4f} | F1={hyb_f1_ecnr:.4f}")
print(f"Delta  | Acc={hyb_acc_ecn - base_acc_ecn:+.4f} | F1={hyb_f1_ecnr - base_f1_ecnr:+.4f}")
print(f"Rescued errors: {rescued}   |   New errors introduced: {hurt}")

print("\nBase Confusion Matrix:")
tn, fp, fn, tp = test_cm_base_ecn.ravel()
print(f"                  Predicted")
print(f"                  Control   PD")
print(f"  Actual Control   {tn:5d}  {fp:5d}")
print(f"  Actual PD        {fn:5d}  {tp:5d}")

print("\nHybrid Confusion Matrix:")
tn, fp, fn, tp = test_cm_hyb_ecn.ravel()
print(f"                  Predicted")
print(f"                  Control   PD")
print(f"  Actual Control   {tn:5d}  {fp:5d}")
print(f"  Actual PD        {fn:5d}  {tp:5d}")

print(f"\n{'='*60}")
print("CLASSIFICATION REPORT (Hybrid, tuned threshold)")
print(f"{'='*60}")
print(classification_report(test_labels_hyb, test_preds_hyb_v, target_names=["Control (0)", "PD (1)"], digits=4))

# Save hybrid ECN bundle
hybrid_save_path = os.path.join(save_dir, "cnn_tcn_hybrid_ecn_parkinsons.pth")
torch.save({
    'base_model_state_dict': best_state,
    'ecn_model_state_dict': ecn_model.state_dict(),
    'config': {'n_channels': cfg.n_channels, 'n_classes': cfg.n_classes,
               'n_samples': cfg.n_samples, 'dropout': cfg.dropout,
               'tcn_channels': cfg.tcn_channels},
    'hybrid_results': {
        'base_threshold': float(base_thr_ecn), 'hybrid_threshold': float(hyb_thr_ecn),
        'base_f1': float(base_f1_ecnr), 'hybrid_f1': float(hyb_f1_ecnr),
        'rescued_errors': rescued, 'new_errors': hurt,
    },
}, hybrid_save_path)
print(f"\nHybrid ECN bundle saved to: {hybrid_save_path}")
